# Guided Planner–ReAct Workflow


<img src="https://www.dailydoseofds.com/content/images/2026/01/https-3a-2f-2fsubstack-post-media-s3-amazonaws-com-2fpublic-2fimages-2f643b6891-84f6-4672-aa1f-4724c5ad2d12_716x526-3.gif" alt="Planning pattern" width="500"/>

This notebook extends `03a`. Instead of the notebook manually supplying the next evidence, `PlanningAgent` assigns an evidence-gathering task and `ReactAgent` carries it out with approved tools.

Keep these two roles separate as you work:
- `PlanningAgent` is the planner: it decides what should happen next.
- `ReactAgent` is the executor: it carries out one evidence-gathering task.

A tool result is an `observation`. `PlanningAgent` reads that observation before assigning the next task. The Python notebook limits `ReactAgent` to approved tools and deliberately stops after two guided rounds.

This notebook is an advanced follow-up to `03a_planning_lab_notebook.ipynb`, not a replacement for it.


## Information Flow and Notebook Structure

`Case question + artifact list -> PlanningAgent creates or updates the plan -> one assigned goal -> ReactAgent uses approved tools -> observation -> PlanningAgent`

The plan is not executed all at once. `PlanningAgent` manages the overall plan; `ReactAgent` executes one assigned goal at a time.

- **Part A — Environment Setup (Steps 1–2):** Load the environment and define the approved tools that `ReactAgent` may use.
- **Part B — Guided Round 1 (Steps 3–5):** `PlanningAgent` creates the initial plan and assigns its first goal. `ReactAgent` uses tools to complete that goal; the tool result becomes the first observation.
- **Part C — Guided Round 2 (Steps 6–7):** `PlanningAgent` receives the first observation, revises the plan, and assigns the next goal. `ReactAgent` completes it and returns the second observation.
- **Part D — Final Report (Step 8):** `PlanningAgent` combines the revised plan and both observations into an evidence-bounded report.

This notebook intentionally shows two rounds. `03c` demonstrates a bounded loop that can continue automatically.


## Lab Question and What to Record

**Case question:** What timeline best explains the missing-phone interval?

**Guided focus:** Use the first `ReactAgent` observation to identify a remaining uncertainty. Then explain how the second `ReactAgent` observation updates the timeline.

Record these four items:

1. **Planner step log:** the first task, the observation it produced, and the next task.
2. **First-observation findings:** the direct timeline evidence in the first `ReactAgent` observation.
3. **Second-observation findings:** the network-related evidence in the second `ReactAgent` observation.
4. **Final timeline and evidence-bounded conclusion:** supported timing and remaining uncertainty.

`NEXT_TASK: ...` is the planner's one-task instruction for a guided round. This notebook shows two rounds; `03c` demonstrates a loop that can continue until `FINISHED:` or its safety limit.

`search_network_evidence(...)` searches only this lab's local `data/` folder, not the web.


## Part A — Environment Setup

### Step 1: Set Up the Notebook

Run this setup cell first. It loads the Lab 4 configuration, checks the notebook location, and prepares the client, imports, and data path.


In [ ]:
# Purpose: This cell supports Step 1: Set Up the Notebook by loading the libraries, settings, and data needed for this section.
import csv
import json
import os
import string
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

# This setup cell assumes you opened the notebook from this lab folder.
# It loads this lab's .env, adds src/ to the import path, and prepares the case data.
LAB_NAME = 'lab4_planning_pattern'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

src_dir = repo_root / 'src'
if str(src_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(src_dir.resolve()))

load_dotenv(env_path, override=True)

MODEL = os.getenv('MODEL')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL')
if not MODEL or not OLLAMA_BASE_URL:
    raise ValueError(f'MODEL or OLLAMA_BASE_URL is missing from {env_path}')

data_dir = lab_dir / 'data'
if not data_dir.exists():
    raise FileNotFoundError('Could not find the Lab 4 data folder')

from agentic_patterns.planning_pattern.planning_agent import PlanningAgent
from agentic_patterns.react_pattern.react_agent import ReactAgent
from agentic_patterns.tool_pattern.tool import tool

client = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print('Data files:', sorted(path.name for path in data_dir.iterdir() if path.is_file()))


### Step 2: Define the `ReactAgent` Tools

These approved tools are `ReactAgent`'s evidence menu:
- `list_artifact_files()`
- `get_incident_window()`
- `get_unlock_events()`
- `get_call_log()`
- `get_whatsapp_events()`
- `search_network_evidence(query)`

All six tools are visible from the start. Good planning still begins with the most direct timeline records; broader search is useful only when an observation leaves a relevant uncertainty, such as connectivity. Each tool returns a short structured summary.


In [ ]:
# Purpose: This cell supports Step 2: Define the `ReactAgent` Tools by defining reusable helper code that performs the work described here.
# Helper function: read one CSV evidence file and return its rows.
def read_csv(filename: str) -> list[dict]:
    with (data_dir / filename).open(encoding='utf-8') as handle:
        return list(csv.DictReader(handle))


# Helper function: return one JSON string with indentation for readability.
def as_pretty_json(payload: dict | list) -> str:
    return json.dumps(payload, indent=2)


# Tool 1: show which evidence files are available in this staged case package.
@tool
def list_artifact_files() -> str:
    """Return the artifact files available in the Lab 4 data folder."""
    artifact_files = sorted(path.name for path in data_dir.iterdir() if path.is_file())
    return as_pretty_json({'artifact_files': artifact_files})


# Tool 2: return the main incident window from the case manifest.
@tool
def get_incident_window() -> str:
    """Return the UTC start and end timestamps for the missing-phone interval."""
    manifest = json.loads((data_dir / 'artifact_manifest.json').read_text(encoding='utf-8'))
    incident_window = manifest['incident_window_utc']
    return as_pretty_json(
        {
            'start': incident_window['start'],
            'end': incident_window['end'],
            'analysis_timezone': manifest['analysis_timezone'],
        }
    )


# Tool 3: return the unlock and lock events that bound device access in the case.
@tool
def get_unlock_events() -> str:
    """Return the recorded unlock and lock events for the device."""
    rows = read_csv('unlock_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 4: return the phone call log summary.
@tool
def get_call_log() -> str:
    """Return the recorded phone call events for the incident period."""
    rows = read_csv('call_log.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 5: return the WhatsApp activity summary.
@tool
def get_whatsapp_events() -> str:
    """Return the WhatsApp activity relevant to the incident period."""
    rows = read_csv('whatsapp_events.csv')
    return as_pretty_json({'event_count': len(rows), 'rows': rows})


# Tool 6: search the local evidence files for network-related clues.
@tool
def search_network_evidence(query: str) -> str:
    """Search the local Lab 4 data folder for network-related evidence that matches the query."""
    query_terms = {
        token.strip(string.punctuation).lower()
        for token in query.split()
        if token.strip(string.punctuation)
    }
    if not query_terms:
        query_terms = {'network'}

    matches = []
    for path in sorted(data_dir.iterdir()):
        if not path.is_file():
            continue
        file_matches = []
        for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
            lower_line = line.lower()
            if any(term in lower_line for term in query_terms):
                file_matches.append({'line_number': line_number, 'text': line})
            if len(file_matches) >= 4:
                break
        if file_matches:
            matches.append({'file': path.name, 'matches': file_matches})

    if not matches:
        fallback_lines = (data_dir / 'network_status.csv').read_text(encoding='utf-8').splitlines()[:4]
        return as_pretty_json(
            {
                'query': query,
                'matches': [],
                'note': 'No exact keyword match found. Showing the start of network_status.csv as a fallback.',
                'fallback_excerpt': fallback_lines,
            }
        )

    return as_pretty_json({'query': query, 'matches': matches})


# All six tools are visible from the start in this simpler teaching version.
react_tools = [
    list_artifact_files,
    get_incident_window,
    get_unlock_events,
    get_call_log,
    get_whatsapp_events,
    search_network_evidence,
]

tool_schema_preview = {
    tool.name: json.loads(tool.fn_signature)
    for tool in react_tools
}

tool_schema_preview


## Part B — Guided Round 1

### Step 3: Let `PlanningAgent` Build the Initial Investigation Plan

This step uses the provided `PlanningAgent` library class to create the initial plan. It can choose a sensible first task from filenames—for example, inspect `unlock_events.csv` to establish device-access timing—but it does not read file contents at this stage.

- **Input — `planner_case_context`:**
  - **`case_question`:** What timeline best explains the missing-phone interval?
  - **`artifact_menu`:** an inventory of available filenames, not file contents.
    - `artifact_manifest.json`
    - `call_log.csv`
    - `chain_of_custody.csv`
    - `network_status.csv`
    - `unlock_events.csv`
    - `whatsapp_events.csv`
- **Output — `initial_plan`:** text returned by the library's `build_initial_plan(...)` method.
  - The plan format is defined in the library's built-in request, not in the notebook code below.
  - That request asks for:
    1. Investigation goal
    2. Ordered steps
    3. Evidence needed for each major step
    4. Replanning triggers

**Design note:** A more capable initial planner could receive a brief description of each artifact—what it records and its time coverage—without receiving every row. That would improve the initial plan while preserving evidence-driven discovery.

Check whether the initial plan starts with the most direct timeline evidence and identifies a useful next uncertainty.


In [ ]:
# Purpose: This cell supports Step 3: Let `PlanningAgent` Build the Initial Investigation Plan by preparing and displaying the initial context before the planner uses it.
planning_agent = PlanningAgent(client=client, model=MODEL)

case_question = 'What timeline best explains the missing-phone interval?'

artifact_menu = {
    'artifact_files': sorted(path.name for path in data_dir.iterdir() if path.is_file()),
}

planner_case_context = (
    f'{case_question}\n\n'
    f'High-level artifact list:\n{json.dumps(artifact_menu, indent=2)}'
)

# Show the exact initial information given to PlanningAgent.
display(Markdown('### `planner_case_context`\n\n```text\n' + planner_case_context + '\n```'))

initial_plan = planning_agent.build_initial_plan(planner_case_context)
display(Markdown('### PlanningAgent Initial Output\n\n' + initial_plan))


### Step 4: Ask `PlanningAgent` for the Next `ReactAgent` Task

- **Input:** `case_question`, `initial_plan`, no `ReactAgent` observations yet, and `react_tools`.
- **Output:** `next_task_1`, the first focused task assigned to `ReactAgent`.

You do not need to follow each helper function in this cell. At a high level, it:

1. Gives the planner the case question, initial plan, no observations yet, and the approved `ReactAgent` tools.
2. Tells the planner to choose one short, tool-supported task, starting with direct timeline evidence.
3. Displays the planner's `NEXT_TASK: ...` decision and saves the task text for `ReactAgent` to carry out in Step 5.

**Example task:**

> Round 1 task: Retrieve the last known unlock events using `get_unlock_events`.

The model's wording may vary, but the task should name a focused evidence check and an approved tool.

`PlanningAgent` chooses the task but does not run tools itself; `ReactAgent` carries out the assigned task in the next step.


In [ ]:
# Purpose: This cell supports Step 4: Ask `PlanningAgent` for the Next `ReactAgent` Task by running the model or agent action and saving its result for review.
PLANNER_CONTROLLER_SYSTEM_PROMPT = """
You coordinate a guided planning workflow for a digital forensics case.

Choose the next single evidence-gathering task for ReactAgent.

Return exactly this format:
NEXT_TASK: <one concrete task sentence>

Rules:
- Ask for only one task at a time.
- This guided notebook always runs two rounds, so provide one next task for each round.
- The task must be solvable with the listed ReactAgent tools.
- Start with the most direct timeline-building evidence before broader search.
- Use network-related search only after direct evidence reveals a remaining dependency or uncertainty about connectivity, delivery timing, or online/offline status.
- Keep the task short and specific.
""".strip()


# Ask PlanningAgent for the next task using the current plan and current observations.
def ask_planner_for_next_task(
    case_question: str,
    current_plan: str,
    observations: str,
    tool_names: list[str],
) -> str:
    observation_text = observations if observations else 'No ReactAgent observations yet.'
    prompt = (
        f'Case question:\n{case_question}\n\n'
        f'Current plan:\n{current_plan}\n\n'
        f'Observations so far:\n{observation_text}\n\n'
        f'Available ReactAgent tools:\n- ' + '\n- '.join(tool_names)
    )
    return client.chat.completions.create(
        messages=[
            {'role': 'system', 'content': PLANNER_CONTROLLER_SYSTEM_PROMPT},
            {'role': 'user', 'content': prompt},
        ],
        model=MODEL,
    ).choices[0].message.content.strip()


# Pull the task text out of a PlanningAgent response such as "NEXT_TASK: ...".
def extract_next_task(planner_output: str) -> str:
    if 'NEXT_TASK:' in planner_output:
        return planner_output.split('NEXT_TASK:', 1)[1].strip()
    return planner_output.strip()


planner_decision_1 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=initial_plan,
    observations='',
    tool_names=[tool.name for tool in react_tools],
)
display(Markdown('### PlanningAgent Decision for ReactAgent Round 1\n\n' + planner_decision_1))

next_task_1 = extract_next_task(planner_decision_1)
print('Round 1 task:', next_task_1)


### Step 5: Run `ReactAgent` on the First Assigned Task

The task assigned from the overall plan is one focused evidence-gathering goal, not necessarily one tool call.

- **Input:**
  - `next_task_1`: the first task selected by `PlanningAgent` in Step 4.
  - `react_tools`: the approved tools that `ReactAgent` may use to complete that task.
- **Output:**
  - `react_agent_result_1`: the first `ReactAgent` observation. It reports the direct evidence found, important timestamps, and any remaining dependency or uncertainty.

**Representative input:**

> `next_task_1`: Retrieve the last known unlock events using `get_unlock_events`.

`ReactAgent` has all approved tools, but for this task it should use `get_unlock_events`.

**Representative output:**

> Direct evidence gathered: `unlock_events.csv` records `device_unlocked` at `2026-02-11T20:55:03Z` and `device_locked` at `2026-02-11T21:25:31Z`.
>
> Remaining dependency or uncertainty: The unlock records establish device-access timing, but do not identify activity that occurred during that interval.

Model wording may vary. `ReactAgent` chooses from the approved tools to answer the assigned task, and its first observation becomes the input to Guided Round 2.


In [ ]:
# Purpose: This cell supports Step 5: Run `ReactAgent` on the First Assigned Task by running the model or agent action and saving its result for review.
round_1_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)

react_agent_prompt_1 = f"""
Carry out this PlanningAgent-assigned task using the available tools:

{next_task_1}

Requirements:
- Focus first on the most direct evidence needed for this assigned task.
- Mention the key timestamps you find.
- If the case still has a remaining dependency or uncertainty, say what it is.
- Return a short response with these labels:
  Direct evidence gathered:
  Remaining dependency or uncertainty:
""".strip()

react_agent_result_1 = round_1_react_agent.run(user_msg=react_agent_prompt_1)
display(Markdown('### ReactAgent Round 1 Final Response\n\n' + react_agent_result_1))


## Part C — Guided Round 2

### Step 6: Feed the First `ReactAgent` Observation Back to `PlanningAgent`

The first `ReactAgent` observation is the result returned in Step 5. It is a real tool result, not a manually supplied record.

- **Input:** `planner_case_context`, `initial_plan`, and `react_agent_result_1`.
- **Output:** `revised_plan_1`, an updated plan, and `next_task_2`, the next focused task for `ReactAgent`.

At a high level, this cell gives `PlanningAgent` the first observation, asks it to revise the plan, then saves its next task decision for Step 7.


In [ ]:
# Purpose: This cell supports Step 6: Feed the First `ReactAgent` Result Back to `PlanningAgent` by preparing or examining the evidence used in this section.
revised_plan_1 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=initial_plan,
    observations=f'ReactAgent round 1 result:\n{react_agent_result_1}',
)
display(Markdown('### PlanningAgent Revised Output After Round 1\n\n' + revised_plan_1))

planner_decision_2 = ask_planner_for_next_task(
    case_question=case_question,
    current_plan=revised_plan_1,
    observations=react_agent_result_1,
    tool_names=[tool.name for tool in react_tools],
)
display(Markdown('### PlanningAgent Decision for ReactAgent Round 2\n\n' + planner_decision_2))

next_task_2 = extract_next_task(planner_decision_2)
print('Round 2 task:', next_task_2)


### Step 7: Run `ReactAgent` for the Second Observation

- **Input:** `next_task_2`, the second task selected in Step 6, and `react_tools`, the approved tools.
- **Output:** `react_agent_result_2`, the second `ReactAgent` observation. It reports the evidence found and its effect on the timeline.

The same approved tools remain available. The planner has read the first `ReactAgent` observation and can choose a task that addresses the remaining uncertainty. Check what evidence the second `ReactAgent` observation adds to the timeline.


In [ ]:
# Purpose: This cell supports Step 7: Run `ReactAgent` for the Second Observation by running the model or agent action and saving its result for review.
round_2_react_agent = ReactAgent(tools=react_tools, client=client, model=MODEL)

react_agent_prompt_2 = f"""
Carry out this PlanningAgent-assigned task using the available tools:

{next_task_2}

Requirements:
- Use whichever available tool best matches the assigned task.
- If the remaining uncertainty is about connectivity, `search_network_evidence(...)` is a reasonable tool to use.
- Report the important file name, timestamps, and status details you find.
- Return a short response with these labels:
  Evidence in the second observation:
  Impact on the timeline:
""".strip()

react_agent_result_2 = round_2_react_agent.run(user_msg=react_agent_prompt_2)
display(Markdown('### ReactAgent Round 2 Final Response\n\n' + react_agent_result_2))


## Part D — Final Report

### Step 8: Let `PlanningAgent` Write the Final Report

This guided notebook stops after two `ReactAgent` observations.

- **Input:** `planner_case_context`, `revised_plan_1`, `react_agent_result_1`, and `react_agent_result_2`.
- **Output:** `revised_plan_2`, the final plan update, and `final_report`, the evidence-bounded report.

The report should show what was checked first, what uncertainty remained, what the second observation added, and what the evidence still cannot establish.


In [ ]:
# Purpose: This cell supports Step 8: Let `PlanningAgent` Write the Final Report by preparing the question, instructions, or context used by the next model step.
combined_observations = (
    'ReactAgent round 1 result:\n'
    f'{react_agent_result_1}\n\n'
    'ReactAgent round 2 result:\n'
    f'{react_agent_result_2}'
)

revised_plan_2 = planning_agent.revise_plan(
    user_msg=planner_case_context,
    current_plan=revised_plan_1,
    observations=combined_observations,
)
display(Markdown('### PlanningAgent Update After Round 2\n\n' + revised_plan_2))

final_report_prompt = (
    'Use the case question, the current investigation plan, and the two ReactAgent observations below to write the final forensic report.\n\n'
    'Return a report with:\n'
    '1. Planner step log\n'
    '2. First-observation findings\n'
    '3. Second-observation findings\n'
    '4. Final timeline and evidence-bounded conclusion\n\n'
    'The second observation comes from the next evidence-gathering task chosen after the first observation.\n\n'
    f'Case question:\n{case_question}\n\n'
    f'Current plan:\n{revised_plan_2}\n\n'
    f'ReactAgent observations:\n{combined_observations}'
)

final_report = planning_agent._complete(final_report_prompt)
display(Markdown('### PlanningAgent Final Report\n\n' + final_report))


## Key Takeaway

`03a_planning_lab_notebook.ipynb` manually supplies the new observation.

This notebook lets `ReactAgent` gather the observation in two guided rounds:
- `PlanningAgent` decides what to do next
- `ReactAgent` gathers evidence with tools
- `PlanningAgent` reads the returned observation and revises the path

`03c_automatic_planner_react_demo.ipynb` repeats the same loop automatically for up to `5` rounds.

Both views teach the same caution: students should not overclaim message delivery when the evidence only supports activity plus later connectivity context.
